# Jev-CeNN — Attention-Free Feasibility Lab: Global + Sliding

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Jev_CeNN_Attention_Free_Global_Sliding_Feasibility_Colab.ipynb)

## The scientific question

**Can a CeNN-style layer replace ModernBERT global attention and sliding attention without retaining any attention operation, while achieving ≥0.99 held-out token-output cosine similarity?**

This is an **experimental feasibility test**, not a proven successful replacement. No numerical results are claimed until the notebook is actually run.

We separately study:
- **Global layer 15:** full-context original teacher attention; the replacement uses fixed multiscale depthwise convolutions and optionally bidirectional cumulative/global average memory.
- **Sliding layer 16:** teacher bidirectional local radius 64 (129 positions); replacement uses the same CeNN candidates but **no sliding SDPA**.
- **Three architectures per layer:** `local` (7/33/65-position conv), `multiscale` (+129-position dilated conv), `memory` (+ whole-sequence prefix/suffix/mean and coarse pooled context).

### Strict definition of attention-free

There is **no** `scaled_dot_product_attention`, `MultiheadAttention`, token-to-token QK similarity, attention scores, tokenwise softmax, kernel/linear attention or landmark attention inside the proposed replacement. The pretrained teacher QKV features are retained, but their q/k streams are used **only pointwise as gates**. Any normalization happens only across a fixed set of CeNN/memory branches at each token. All token-mixing paths have linear cost in sequence length.

### Honest evaluation

1. **Original-teacher replay** with real ModernBERT position IDs and masks (must pass).
2. **Attention-API interdiction:** monkeypatch `torch.nn.functional.scaled_dot_product_attention` in each candidate preflight so calling it immediately fails.
3. Same train/probe/audit teacher input/output cache per layer; independent cases and no official test data used.
4. Distill for the same optimizer updates; choose architecture by **held-out probe score only**. Report all candidates' separate audit results, valid-token cosine, NMSE and measured CUDA layer latency.
5. Insert only the selected layer in a clean student and measure downstream teacher decision agreement and gold-label accuracy on DEV, separately for global and sliding. Do not infer that excellent local cosine guarantees full-model fidelity.
6. Optional further refinement of the probe winner (checkpoint from probe, independent audit again). Summarize **PASS / NOT REACHED** for ≥0.99 local cosine and ≥0.99 downstream teacher agreement.

**Controls:** Native teacher replay (cos=1) and source-layer latency are measured but not considered attention-free candidates. The official test split stays untouched in this exploratory stage. Passing one layer does not establish that replacing all 30 will pass.

**Practical limit:** GPU training cost is ~6×SCREEN_STEPS plus optional refinement; run on a CUDA Colab runtime. Colab scratch checkpoints do not persist across session restarts unless `OUTPUT_DIR` is set to a mounted Drive directory.


In [ ]:
#@title 1. Install required pinned runtime
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Load original teacher, isolate layers, configure independent evaluation
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv, _valid_tokens
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import require_finite
from tinycenn_lm.jev_cenn_free import JevCeNNFreeMixer

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions" #@param {type:"string"}
GLOBAL_LAYER=15 #@param {type:"integer"}
SLIDING_LAYER=16 #@param {type:"integer"}
VARIANTS=("local","multiscale","memory")
SCREEN_STEPS=250 #@param {type:"integer"}
REFINE_WINNER=False #@param {type:"boolean"}
REFINE_STEPS=600 #@param {type:"integer"}
COARSE_STRIDE=32 #@param {type:"integer"}
TRAIN_CASES=600 #@param {type:"integer"}
DEV_CASES=180 #@param {type:"integer"}
MAX_LEN=512 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES=32 #@param {type:"integer"}
PROBE_CACHE_BATCHES=8 #@param {type:"integer"}
AUDIT_CACHE_BATCHES=8 #@param {type:"integer"}
CHECK_EVERY=50
TARGET_COSINE=0.99
TARGET_NMSE=0.025
TARGET_AGREEMENT=0.99
OUTPUT_DIR=Path("/content/Jev_CeNN_Attention_Free_Feasibility")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
STEPS=SCREEN_STEPS
LAYER=GLOBAL_LAYER
assert SCREEN_STEPS>=1 and REFINE_STEPS>=1 and BATCH>=1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU (Colab > Runtime > Change runtime type).")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

print("GPU:", torch.cuda.get_device_name(0), "| AMP:", amp_dtype)
GLOBAL_LAYERS = [i for i, layer in enumerate(teacher.encoder.layers)
                 if tuple(layer.attn.local_attention) == (-1, -1)]
SLIDING_LAYERS = [i for i, layer in enumerate(teacher.encoder.layers)
                  if tuple(layer.attn.local_attention) != (-1, -1)]
assert SLIDING_LAYERS and GLOBAL_LAYERS
NATIVE_RADIUS = teacher.encoder.layers[SLIDING_LAYER].attn.local_attention[0]
assert GLOBAL_LAYER in GLOBAL_LAYERS and SLIDING_LAYER in SLIDING_LAYERS
print("Global teacher layer:", GLOBAL_LAYER, "| Sliding teacher layer:", SLIDING_LAYER)
print("Global layers untouched by default:", GLOBAL_LAYERS)
print("Sliding layers to replace:", SLIDING_LAYERS)
print("Teacher:", SOURCE_MODEL, "| official test split stays untouched")


In [ ]:
#@title 3. Disjoint train/probe/audit/DEV data and teacher I/O
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, True)

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
        # ModernBERT SDPA requires position_ids and sliding_window_mask.
        # Replay the exact teacher call, including all positional/mask kwargs.
        got["call_kwargs"] = {
            key: (value.detach() if torch.is_tensor(value) else value)
            for key, value in kwargs.items()
        }
        if "position_ids" not in got["call_kwargs"] or "sliding_window_mask" not in got["call_kwargs"]:
            raise RuntimeError("Teacher attention hook did not capture ModernBERT SDPA kwargs")
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred = repl(entry["x"], **entry["call_kwargs"])[0]
            require_finite(pred, "V3 sliding local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.50 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/dev decisions:", len(train_items), len(dev_items))
assert len(dev_items) >= BATCH * (PROBE_CACHE_BATCHES + AUDIT_CACHE_BATCHES), "Increase DEV_CASES for disjoint probe and audit caches"
print("official test remains untouched; probe/audit I/O non-overlapping")


In [ ]:
#@title 4. Native teacher replay; prohibit ALL attention APIs during student preflight
import inspect
from unittest.mock import patch
assert len(VARIANTS)==3
DEV_QUALITY = dev_items[(PROBE_CACHE_BATCHES+AUDIT_CACHE_BATCHES)*BATCH:]
assert len(DEV_QUALITY)>=120
def build_free(layer_idx,variant):
    return JevCeNNFreeMixer(
        teacher.encoder.layers[layer_idx].attn,variant=variant,
        coarse_stride=COARSE_STRIDE
    ).to(device)
def no_attention(*args,**kwargs):
    raise AssertionError("Forbidden SDPA / attention called by Jev-CeNN!")
def preflight(kind,idx):
    selected=batch(dev_items,0,min(BATCH,2))
    teacher_entry=attn_io(teacher,idx,selected)
    with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
        ref=teacher.encoder.layers[idx].attn(
            teacher_entry["x"],**teacher_entry["call_kwargs"]
        )[0]
    torch.testing.assert_close(ref.float(),teacher_entry["y"].float(),
                               atol=1e-3,rtol=1e-3)
    print(f"TEACHER REPLAY {kind} layer={idx}: PASS")
    for variant in VARIANTS:
        candidate=build_free(idx,variant).train()
        candidate.out_drop.eval()
        # If the candidate executes even one SDPA operation, fail immediately.
        with patch.object(F,"scaled_dot_product_attention",side_effect=no_attention):
            with torch.autocast("cuda",dtype=amp_dtype):
                prediction=candidate(
                    teacher_entry["x"],**teacher_entry["call_kwargs"]
                )[0]
        assert prediction.shape==teacher_entry["y"].shape
        require_finite(prediction,f"free {kind}/{variant} preflight")
        mask=selected["attention_mask"].bool()
        criterion=(prediction.float()[mask]-teacher_entry["y"].float()[mask]).square().mean()
        grads=torch.autograd.grad(
            criterion,[p for p in candidate.parameters() if p.requires_grad],
            allow_unused=True
        )
        assert any(g is not None and bool(torch.isfinite(g).all())
                   and bool(g.abs().sum()>0) for g in grads),(
            f"Disconnected candidate {kind}/{variant}"
        )
        print(f"  {kind}/{variant}: ATTENTION-FREE forward + gradient PASS | "
              f"initial MSE={float(criterion):.6f}")
        del candidate,prediction,grads
    torch.cuda.empty_cache()
preflight("global",GLOBAL_LAYER)
preflight("sliding",SLIDING_LAYER)


In [ ]:
#@title 5. Fixed-update attention-free distillation and probe-only checkpoints
def score_local(m):
    return m["nmse"] + 1.50 * (1.0 - m["cosine"])

def optimizer_v28(model):
    core, output = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (output if name.startswith("Wo.") else core).append(p)
    params = core + output
    opt = torch.optim.AdamW([
        {"params": core, "lr": 0.001, "base_lr": 0.001},
        {"params": output, "lr": 0.00002, "base_lr": 0.00002},
    ], betas=(0.9, 0.95), weight_decay=2e-4)
    return opt, params

def train_400(model, train_cache, probe_cache, window):
    optimizer, params = optimizer_v28(model)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16, init_scale=128.0)
    best_metrics = local_metrics_from_cache(model, probe_cache)
    best_score = score_local(best_metrics)
    best_step = 0
    best_state = state_cpu(model)
    history = [{"step": 0, **best_metrics}]
    successes, attempts = 0, 0
    print(f"W{window} step    0/{STEPS}: NMSE={best_metrics['nmse']:.5f} cosine={best_metrics['cosine']:.5f}")

    while successes < STEPS:
        if attempts > STEPS + 300:
            raise RuntimeError("Too many AMP overflow/skipped optimizer steps")
        # A moderate cosine decay + short warmup is more stable than PDelta's
        # former 1e-2 opening learning rate, especially with trainable temperature.
        warm = min(1.0, (successes + 1) / 15.0)
        decay = 0.15 + 0.85 * 0.5 * (1.0 + math.cos(math.pi * successes / STEPS))
        for group in optimizer.param_groups:
            group["lr"] = group["base_lr"] * warm * decay

        model.train()
        model.out_drop.eval()
        entry = train_cache[attempts % len(train_cache)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            output = model(entry["x"], **entry["call_kwargs"])[0]
            m = entry["valid"][:, :, None].float()
            target = entry["y"].float()
            pred = output.float()
            mse = (((pred - target) ** 2) * m).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            power = ((target ** 2) * m).sum() / (m.sum() * pred.shape[-1]).clamp_min(1)
            nmse = mse / power.clamp_min(1e-8)
            cos = F.cosine_similarity(pred[entry["valid"]], target[entry["valid"]], dim=-1).mean()
            loss = nmse + 1.50 * (1.0 - cos)
        require_finite(loss, f"V28 W{window} training loss")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        attempts += 1
        if scaler.is_enabled() and scaler.get_scale() < old_scale:
            continue
        successes += 1
        if successes == 1 or successes % CHECK_EVERY == 0 or successes == STEPS:
            probe = local_metrics_from_cache(model, probe_cache)
            history.append({"step": successes, **probe})
            s = score_local(probe)
            if s < best_score:
                best_score, best_metrics, best_step = s, probe, successes
                best_state = state_cpu(model)
            print(
                f"W{window} step {successes:4d}/{STEPS}: "
                f"NMSE={probe['nmse']:.5f} cosine={probe['cosine']:.5f} "
                f"| best cosine={best_metrics['cosine']:.5f} @ {best_step}"
            )
    model.load_state_dict(best_state, strict=True)
    model.eval()
    return {
        "best_step": best_step, "best_probe": best_metrics,
        "attempts": attempts, "successful_updates": successes,
        "history": history,
    }


In [ ]:
#@title 6. Held-out whole-model decision/latency evaluation
@torch.no_grad()
def validate_pair(student, items=dev_items, limit=None):
    teacher.eval()
    student.eval()
    limit = len(items) if limit is None else min(int(limit), len(items))
    total = agree = teacher_gold = student_gold = 0
    js = []
    for off in range(0, limit, BATCH):
        chosen = items[off:min(off + BATCH, limit)]
        b = direct_batch(chosen)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b)
            sl, _ = student(**b)
        require_finite(tl, "teacher validation logits")
        require_finite(sl, "SlidingFusion validation logits")
        allowed = b["marker_mask"].bool()
        tp = torch.softmax(tl.float().masked_fill(~allowed, -1e9), -1)
        sp = torch.softmax(sl.float().masked_fill(~allowed, -1e9), -1)
        gt = torch.tensor([row["gold_index"] for row in chosen], device=device)
        agree += int((tp.argmax(-1) == sp.argmax(-1)).sum())
        teacher_gold += int((tp.argmax(-1) == gt).sum())
        student_gold += int((sp.argmax(-1) == gt).sum())
        mid = 0.5 * (tp + sp)
        q = 0.5 * (tp * (tp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())).sum(-1)
        q += 0.5 * (sp * (sp.clamp_min(1e-8).log() - mid.clamp_min(1e-8).log())).sum(-1)
        js.extend(q.cpu().tolist())
        total += len(chosen)
    return dict(
        decisions=total, agreement=agree / max(total, 1),
        mean_js=float(np.mean(js)),
        teacher_gold_accuracy=teacher_gold / max(total, 1),
        student_gold_accuracy=student_gold / max(total, 1),
    )

@torch.inference_mode()
def timed_attention(module, entry, warmup=5, repeats=15):
    module.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        for _ in range(warmup):
            module(entry["x"], **entry["call_kwargs"])
        torch.cuda.synchronize()
        ms = []
        for _ in range(repeats):
            start = time.perf_counter()
            module(entry["x"], **entry["call_kwargs"])
            torch.cuda.synchronize()
            ms.append((time.perf_counter() - start) * 1000)
    return float(np.median(ms))

def attention_coverage(cache, window):
    lengths = torch.cat([item["valid"].sum(-1).cpu() for item in cache]).numpy()
    # A centered W=96 attends only 47 tokens left and 48 right.
    # Length <= W alone does NOT guarantee all-to-all attention.
    full_context_limit = min((window - 1) // 2, window // 2) + 1
    return {
        "sequences": int(len(lengths)),
        "median_valid_tokens": float(np.median(lengths)),
        "max_valid_tokens": int(np.max(lengths)),
        "fraction_fully_covered_by_local_window": float(np.mean(lengths <= full_context_limit)),
        "full_context_length_limit": int(full_context_limit),
        "sequences_longer_than_window": int(np.sum(lengths > window)),
        "sequences_at_least_2x_window": int(np.sum(lengths >= 2 * window)),
    }



In [ ]:
#@title 7. Blind train/probe/audit tournament: 3 candidates × 2 attention types
RESULTS=[]
HISTORY={}
for kind,idx in (("global",GLOBAL_LAYER),("sliding",SLIDING_LAYER)):
    print("\n"+"="*95)
    print("ATTENTION-FREE TOURNAMENT:",kind,"layer",idx)
    print("="*95)
    tr=cache_teacher_io(idx,train_items,TRAIN_CACHE_BATCHES,offset=idx*23)
    probe=cache_teacher_io(idx,dev_items,PROBE_CACHE_BATCHES,offset=0)
    audit=cache_teacher_io(
        idx,dev_items,AUDIT_CACHE_BATCHES,
        offset=PROBE_CACHE_BATCHES*BATCH
    )
    original=teacher.encoder.layers[idx].attn
    # Measure selection latency on PROBE, not on the held-out AUDIT.
    teacher_ms=timed_attention(original,probe[0],warmup=5,repeats=12)
    for variant in VARIANTS:
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
        candidate=build_free(idx,variant)
        initial=local_metrics_from_cache(candidate,probe)
        STEP_LABEL=f"{kind}_L{idx}_{variant}"
        file=OUTPUT_DIR/f"{STEP_LABEL}.pt"
        if file.exists():
            payload=torch.load(file,map_location="cpu",weights_only=True)
            assert payload["layer"]==idx and payload["variant"]==variant
            candidate.load_state_dict(payload["state_dict"],strict=True)
            info=payload["training"]
            print("Checkpoint reused:",file)
        else:
            STEPS=SCREEN_STEPS
            info=train_400(candidate,tr,probe,variant)
            torch.save({
                "architecture":"jev_cenn_free_mixer_v1",
                "layer":idx,"kind":kind,"variant":variant,
                "state_dict":state_cpu(candidate),"training":info
            },file)
        measured=local_metrics_from_cache(candidate,audit)
        candidate_ms=timed_attention(candidate,probe[0],warmup=5,repeats=12)
        # End-to-end metrics are diagnostic only, never used in architecture selection.
        original_student_layer=student_eval.encoder.layers[idx].attn
        student_eval.encoder.layers[idx].attn=candidate.eval().requires_grad_(False)
        try:
            quality=validate_pair(student_eval,DEV_QUALITY,limit=120)
        finally:
            student_eval.encoder.layers[idx].attn=original_student_layer
        row={
            "kind":kind,"layer":idx,"variant":variant,
            "initial_probe":initial,"probe":info["best_probe"],
            "audit":measured,"train_step":info["best_step"],
            "local_attention_free":True,
            "teacher_layer_ms":teacher_ms,
            "student_layer_ms":candidate_ms,
            "layer_speedup":teacher_ms/max(candidate_ms,1e-9),
            "dev_e2e":quality,
            "probe_pass":(info["best_probe"]["cosine"]>=TARGET_COSINE
                          and info["best_probe"]["nmse"]<=TARGET_NMSE),
            "audit_pass":(measured["cosine"]>=TARGET_COSINE
                         and measured["nmse"]<=TARGET_NMSE),
            "checkpoint":str(file)
        }
        RESULTS.append(row)
        HISTORY[STEP_LABEL]=info["history"]
        print(f"{kind}/{variant}: "
              f"probe cos={row['probe']['cosine']:.5f} "
              f"blind audit cos={row['audit']['cosine']:.5f} "
              f"NMSE={row['audit']['nmse']:.5f} "
              f"teacher-agreement={quality['agreement']:.4f} "
              f"speedup={row['layer_speedup']:.3f}x "
              f"**AUDIT PASS={row['audit_pass']}**")
        del candidate
        torch.cuda.empty_cache()
    del tr,probe,audit
    torch.cuda.empty_cache()


In [ ]:
#@title 8. Select by PROBE only and optionally refine most promising architecture
# If any candidate reaches 0.99 on probe, select the fastest eligible;
# otherwise choose the lowest probe NMSE+cosine score.
WINNERS={}
FINAL={}
for kind,idx in (("global",GLOBAL_LAYER),("sliding",SLIDING_LAYER)):
    options=[row for row in RESULTS if row["kind"]==kind]
    good=[row for row in options if row["probe_pass"]]
    winner=(min(good,key=lambda x:x["student_layer_ms"]) if good else
            min(options,key=lambda x:score_local(x["probe"])))
    WINNERS[kind]=winner
    print(f"SELECTED {kind}: {winner['variant']} "
          f"PROBE cos={winner['probe']['cosine']:.6f}, "
          f"AUDIT cos={winner['audit']['cosine']:.6f} "
          f"(audit NOT used to select)")
    if REFINE_WINNER:
        tc=cache_teacher_io(idx,train_items,TRAIN_CACHE_BATCHES,offset=idx*23)
        pc=cache_teacher_io(idx,dev_items,PROBE_CACHE_BATCHES)
        ac=cache_teacher_io(
            idx,dev_items,AUDIT_CACHE_BATCHES,offset=PROBE_CACHE_BATCHES*BATCH
        )
        candidate=build_free(idx,winner["variant"])
        ckpt=torch.load(winner["checkpoint"],map_location="cpu",weights_only=True)
        candidate.load_state_dict(ckpt["state_dict"],strict=True)
        STEPS=REFINE_STEPS
        refine=train_400(candidate,tc,pc,winner["variant"])
        blind=local_metrics_from_cache(candidate,ac)
        newfile=OUTPUT_DIR/f"refined_{kind}_L{idx}_{winner['variant']}.pt"
        torch.save({
            "layer":idx,"variant":winner["variant"],
            "state_dict":state_cpu(candidate),"train":refine,
            "audit":blind
        },newfile)
        current=dict(winner)
        current["refined_audit"]=blind
        current["refined_checkpoint"]=str(newfile)
        current["refined_probe"]=refine["best_probe"]
        final_item=current
        del candidate,tc,pc,ac
    else:
        final_item=dict(winner)
        final_item["refined_audit"]=winner["audit"]
        final_item["refined_probe"]=winner["probe"]
        final_item["refined_checkpoint"]=winner["checkpoint"]
    FINAL[kind]=final_item
    torch.cuda.empty_cache()


In [ ]:
#@title 9. Final independent DEV quality, actual CUDA speed, truthful feasibility verdict
REPORT={"architecture":"Jev-CeNN attention-free (single-layer feasibility)",
        "source_teacher":SOURCE_MODEL,"target_cosine":TARGET_COSINE,
        "target_nmse":TARGET_NMSE,"target_agreement":TARGET_AGREEMENT,
        "screen_steps":SCREEN_STEPS,
        "training_uses_official_test":False,
        "architectures":list(VARIANTS),
        "all_candidates":RESULTS,
        "winners":{}}
for kind,idx in (("global",GLOBAL_LAYER),("sliding",SLIDING_LAYER)):
    item=FINAL[kind]
    model=build_free(idx,item["variant"])
    ckpt=torch.load(item["refined_checkpoint"],map_location="cpu",weights_only=True)
    model.load_state_dict(ckpt["state_dict"],strict=True)
    model.eval().requires_grad_(False)
    original=student_eval.encoder.layers[idx].attn
    student_eval.encoder.layers[idx].attn=model
    try:
        e2e=validate_pair(student_eval,DEV_QUALITY)
    finally:
        student_eval.encoder.layers[idx].attn=original
    metrics=item["refined_audit"]
    fidelity_gate=(metrics["cosine"]>=TARGET_COSINE
                   and metrics["nmse"]<=TARGET_NMSE)
    decision_gate=e2e["agreement"]>=TARGET_AGREEMENT
    verdict={
        "layer":idx,"variant":item["variant"],
        "probe":item["refined_probe"],
        "blind_audit":metrics,
        "full_dev_e2e":e2e,
        "attention_free":True,
        "fidelity_gate_pass":bool(fidelity_gate),
        "decision_gate_pass":bool(decision_gate),
        "all_gates_pass":bool(fidelity_gate and decision_gate),
        "checkpoint":item["refined_checkpoint"]
    }
    REPORT["winners"][kind]=verdict
    print(f"\n{kind.upper()} FINAL attention-free "
          f"cos={metrics['cosine']:.6f} "
          f"NMSE={metrics['nmse']:.6f} "
          f"decision-agreement={e2e['agreement']:.6f} "
          f"gold={e2e['student_gold_accuracy']:.6f} "
          f"0.99 gates={'PASS' if verdict['all_gates_pass'] else 'NOT REACHED'}")
    del model
REPORT["both_single_layer_gates_pass"]=all(
    x["all_gates_pass"] for x in REPORT["winners"].values()
)
report_file=OUTPUT_DIR/"jev_cenn_attention_free_feasibility.json"
report_file.write_text(json.dumps(REPORT,indent=2,allow_nan=False))
print("\nRESULT:",report_file)
print("Both independent single-layer feasibility gates:",
      "PASS" if REPORT["both_single_layer_gates_pass"] else "NOT REACHED")
print("REMINDER: this does NOT test all-30 attention-free conversion.")


In [ ]:
#@title 10. Visualize blind audits, actual speed and cumulative evidence
import matplotlib.pyplot as plt
labels=[f"{r['kind']}\n{r['variant']}" for r in RESULTS]
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
axes[0].bar(labels,[r["audit"]["cosine"] for r in RESULTS])
axes[0].axhline(TARGET_COSINE,color="gray",ls="--",label="0.99 goal")
axes[0].set(title="Blind audit: mean token cosine",
            ylabel="Cosine",ylim=(0,1.02))
axes[0].legend()
axes[1].bar(labels,[r["layer_speedup"] for r in RESULTS])
axes[1].axhline(1,color="gray",ls="--",label="1×")
axes[1].set(title="GPU layer speedup over native attention",
            ylabel="Teacher / student latency")
for ax in axes:
    ax.tick_params(axis="x",labelrotation=38)
    ax.grid(axis="y",alpha=.2)
fig.tight_layout()
fig_path=OUTPUT_DIR/"jev_cenn_attention_free_screen.png"
fig.savefig(fig_path,dpi=160,bbox_inches="tight")
plt.show()
print("Plot:",fig_path)
